In [1]:
import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet", "--force-reinstall", "--no-cache-dir",
    "torch==2.7.1",
    "torchvision==0.22.1",
    "torchaudio==2.7.1",
    "transformers==4.52.4",
    "accelerate==1.7.0",
    "datasets==3.6.0",
    "evaluate==0.4.3",
    "tokenizers==0.21.1"
])


0

In [1]:
import sys
import transformers
import accelerate
import datasets
import torch

print("Python executable:", sys.executable)
print("Transformers:", transformers.__version__, transformers.__file__)
print("Accelerate:", accelerate.__version__, accelerate.__file__)
print("Datasets:", datasets.__version__)
print("Torch:", torch.__version__)

Python executable: /usr/bin/python3
Transformers: 4.52.4 /usr/local/lib/python3.12/dist-packages/transformers/__init__.py
Accelerate: 1.7.0 /usr/local/lib/python3.12/dist-packages/accelerate/__init__.py
Datasets: 3.6.0
Torch: 2.7.1+cu126


In [2]:
# =========================================================
# 1. IMPORTS
# =========================================================
import os
import csv
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from datasets import load_dataset, Dataset
from sklearn.metrics import f1_score, classification_report
from scipy.stats import pearsonr

from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    DistilBertModel,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    TrainerCallback
)



In [ ]:
# =========================================================
# 1. IMPORTS
# =========================================================
import os
import csv
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from datasets import load_dataset, Dataset
from sklearn.metrics import f1_score, classification_report
from scipy.stats import pearsonr

from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    RobertaModel,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    TrainerCallback
)



In [ ]:
# =========================================================
# 2. GOOGLE DRIVE + SETUP
# =========================================================
from google.colab import drive
drive.mount("/content/drive")

SEED = 42
K = 5
MODEL_NAME = "roberta-base"

OUTPUT_DIR = "/content/drive/MyDrive/RoBERTa_Hierarchical_ManualKFold"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)




In [ ]:
# =========================================================
# 3. LOAD DATASET
# =========================================================
train_data = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="train"
)

val_data = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="dev"
)

test_data = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="test"
)

train_df = train_data.to_pandas()
val_df = val_data.to_pandas()
test_df = test_data.to_pandas()

print("Original sizes:")
print({
    "train": len(train_df),
    "dev": len(val_df),
    "test": len(test_df)
})


# =========================================================
# 4. LABELS
# =========================================================
EMOTIONS = ["anger", "fear", "joy", "sadness", "surprise"]
INTENSITY_COLUMNS = [f"{emotion}_intensity" for emotion in EMOTIONS]
LEVELS = [1, 2, 3]

LABELS = [
    f"{emotion}_{level}"
    for emotion in EMOTIONS
    for level in LEVELS
]


# =========================================================
# 5. PREPARE TWO-STEP DATA
# =========================================================
def prepare_two_step_data(df):
    df = df.copy()

    if "disgust" in df.columns:
        df = df.drop(columns=["disgust"])

    for emotion in EMOTIONS:
        df[f"{emotion}_intensity"] = df[emotion].astype(int)

    for emotion in EMOTIONS:
        df[emotion] = (df[f"{emotion}_intensity"] > 0).astype(int)

    return df[["text"] + EMOTIONS + INTENSITY_COLUMNS]


train_two = prepare_two_step_data(train_df)
val_two = prepare_two_step_data(val_df)
test_two = prepare_two_step_data(test_df)


# =========================================================
# 6. 70/20/10-STYLE SETUP FOR 5-FOLD CV
# =========================================================
full_df = pd.concat(
    [train_two, val_two, test_two],
    ignore_index=True
)

full_df = full_df[["text"] + EMOTIONS + INTENSITY_COLUMNS]

full_df = full_df.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

n = len(full_df)

test_start = int(0.9 * n)

cv_df = full_df[:test_start].reset_index(drop=True)
test_final_df = full_df[test_start:].reset_index(drop=True)

print("\n70/20/10-style cross-validation setup:")
print({
    "full_data": len(full_df),
    "cv_data_90_percent": len(cv_df),
    "test_data_10_percent": len(test_final_df)
})

print("\nIn each 5-fold iteration:")
print({
    "train_per_fold_approx": int(0.8 * len(cv_df)),
    "val_per_fold_approx": int(0.2 * len(cv_df)),
    "test_fixed": len(test_final_df)
})


# =========================================================
# 7. TOKENIZER
# =========================================================
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128
    )


# =========================================================
# 8. DATASET PREPARATION FUNCTIONS
# =========================================================
def add_step1_labels(example):
    example["labels"] = [
        float(example[emotion])
        for emotion in EMOTIONS
    ]

    return example


def add_step2_labels(example):
    example["labels"] = [
        int(example[col])
        for col in INTENSITY_COLUMNS
    ]

    return example


def prepare_step1_dataset(df):
    step1_df = df[["text"] + EMOTIONS].copy()

    dataset = Dataset.from_pandas(
        step1_df,
        preserve_index=False
    )

    dataset = dataset.map(
        tokenize_function,
        batched=True
    )

    dataset = dataset.map(add_step1_labels)

    dataset.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "labels"]
    )

    return dataset


def prepare_step2_dataset(df):
    step2_df = df[["text"] + INTENSITY_COLUMNS].copy()

    dataset = Dataset.from_pandas(
        step2_df,
        preserve_index=False
    )

    dataset = dataset.map(
        tokenize_function,
        batched=True
    )

    dataset = dataset.map(add_step2_labels)

    dataset.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "labels"]
    )

    return dataset


test_step1_ds = prepare_step1_dataset(test_final_df)
test_step2_ds = prepare_step2_dataset(test_final_df)


# =========================================================
# 9. CREATE MANUAL FOLDS
# =========================================================
def create_manual_folds(cv_df, k=5, seed=42):
    n = len(cv_df)

    indices = np.arange(n)

    np.random.seed(seed)
    np.random.shuffle(indices)

    fold_sizes = np.full(k, n // k, dtype=int)
    fold_sizes[:n % k] += 1

    folds = []
    current = 0

    for fold_size in fold_sizes:
        start = current
        stop = current + fold_size

        folds.append(indices[start:stop])

        current = stop

    fold_data = {}

    for fold in range(k):
        fold_number = fold + 1

        val_idx = folds[fold]

        train_idx = np.concatenate([
            folds[i]
            for i in range(k)
            if i != fold
        ])

        fold_train_df = cv_df.iloc[train_idx].reset_index(drop=True)
        fold_val_df = cv_df.iloc[val_idx].reset_index(drop=True)

        fold_data[fold_number] = {
            "train_df": fold_train_df,
            "val_df": fold_val_df
        }

    return fold_data


fold_data = create_manual_folds(
    cv_df=cv_df,
    k=K,
    seed=SEED
)


# =========================================================
# 10. CLASS COUNTS + PLOT
# =========================================================
def get_15_label_counts(df):
    counts = {}

    for emotion in EMOTIONS:
        for level in LEVELS:
            label = f"{emotion}_{level}"
            counts[label] = int(
                (df[f"{emotion}_intensity"] == level).sum()
            )

    return pd.Series(counts)


def print_fold_class_counts(fold_number):
    fold_train_df = fold_data[fold_number]["train_df"]
    fold_val_df = fold_data[fold_number]["val_df"]

    train_counts = get_15_label_counts(fold_train_df)
    val_counts = get_15_label_counts(fold_val_df)

    counts_df = pd.DataFrame({
        "train_count": train_counts,
        "val_count": val_counts
    })

    print("\n================================")
    print(f"Fold {fold_number}")
    print("================================")
    print("Train samples:", len(fold_train_df))
    print("Validation samples:", len(fold_val_df))

    print("\nTrain and validation class counts:")
    print(counts_df)

    return counts_df


def plot_selected_fold(fold_number):
    fold_train_df = fold_data[fold_number]["train_df"]
    fold_val_df = fold_data[fold_number]["val_df"]

    train_counts = get_15_label_counts(fold_train_df)
    val_counts = get_15_label_counts(fold_val_df)

    x = np.arange(len(LABELS))
    width = 0.35

    plt.figure(figsize=(16, 7))

    train_bars = plt.bar(
        x - width / 2,
        train_counts.values,
        width,
        label="Train"
    )

    val_bars = plt.bar(
        x + width / 2,
        val_counts.values,
        width,
        label="Validation"
    )

    for bar in train_bars:
        height = bar.get_height()

        plt.text(
            bar.get_x() + bar.get_width() / 2,
            height,
            str(int(height)),
            ha="center",
            va="bottom",
            fontsize=8,
            rotation=90
        )

    for bar in val_bars:
        height = bar.get_height()

        plt.text(
            bar.get_x() + bar.get_width() / 2,
            height,
            str(int(height)),
            ha="center",
            va="bottom",
            fontsize=8,
            rotation=90
        )

    plt.xlabel("Emotion-intensity labels")
    plt.ylabel("Number of samples")

    plt.title(
        f"Fold {fold_number} Class Distribution\n"
        f"Train={len(fold_train_df)}, Validation={len(fold_val_df)}"
    )

    plt.xticks(
        x,
        LABELS,
        rotation=45
    )

    plt.legend()
    plt.tight_layout()
    plt.show()
    plt.close()


def show_all_folds():
    for fold_number in range(1, K + 1):
        print_fold_class_counts(fold_number)
        plot_selected_fold(fold_number)


show_all_folds()


# =========================================================
# 11. STEP 1 METRICS
# =========================================================
def compute_metrics_step1(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)

    f1_macro = f1_score(
        labels,
        preds,
        average="macro",
        zero_division=0
    )

    f1_micro = f1_score(
        labels,
        preds,
        average="micro",
        zero_division=0
    )

    pearsons = []

    for i in range(labels.shape[1]):
        if np.std(labels[:, i]) == 0 or np.std(probs[:, i]) == 0:
            pearsons.append(0.0)
        else:
            p, _ = pearsonr(labels[:, i], probs[:, i])
            pearsons.append(0.0 if np.isnan(p) else float(p))

    return {
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
        "pearson_mean": float(np.mean(pearsons))
    }


# =========================================================
# 12. STEP 2 MODEL
# =========================================================
class RobertaStep2IntensityModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.roberta = RobertaModel.from_pretrained(MODEL_NAME)

        hidden_size = self.roberta.config.hidden_size

        self.dropout = nn.Dropout(0.1)

        self.classifier = nn.Linear(
            hidden_size,
            len(EMOTIONS) * 4
        )

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)

        logits = self.classifier(cls_output)

        logits = logits.view(
            -1,
            len(EMOTIONS),
            4
        )

        loss = None

        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()

            loss = loss_fct(
                logits.view(-1, 4),
                labels.view(-1)
            )

        return {
            "loss": loss,
            "logits": logits
        }


# =========================================================
# 13. STEP 2 METRICS
# =========================================================
def compute_metrics_step2(eval_pred):
    logits, labels = eval_pred

    preds = np.argmax(logits, axis=-1)

    true_flat = labels.reshape(-1)
    pred_flat = preds.reshape(-1)

    f1_macro = f1_score(
        true_flat,
        pred_flat,
        average="macro",
        zero_division=0
    )

    f1_micro = f1_score(
        true_flat,
        pred_flat,
        average="micro",
        zero_division=0
    )

    if np.std(true_flat) == 0 or np.std(pred_flat) == 0:
        pearson_mean = 0.0
    else:
        pearson_mean, _ = pearsonr(true_flat, pred_flat)
        pearson_mean = 0.0 if np.isnan(pearson_mean) else float(pearson_mean)

    return {
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
        "pearson_mean": pearson_mean
    }


# =========================================================
# 14. CUSTOM EARLY STOPPING CALLBACK
# =========================================================
class StopOnValLossAfterMinEpoch(TrainerCallback):
    def __init__(self, min_epoch=3, patience=1):
        self.min_epoch = min_epoch
        self.patience = patience
        self.best_loss = None
        self.bad_epochs = 0

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is None:
            return control

        epoch = int(round(float(state.epoch)))
        val_loss = metrics.get("eval_loss")

        if val_loss is None:
            return control

        if self.best_loss is None or val_loss < self.best_loss:
            self.best_loss = val_loss
            self.bad_epochs = 0
        else:
            if epoch >= self.min_epoch:
                self.bad_epochs += 1

        if epoch >= self.min_epoch and self.bad_epochs >= self.patience:
            print(
                f"\nEarly stopping at epoch {epoch}. "
                f"Best validation loss: {self.best_loss:.6f}"
            )
            control.should_training_stop = True

        return control


# =========================================================
# 15. SIMPLE LOGGER CALLBACK
# =========================================================
class SaveEpochLogger(TrainerCallback):
    def __init__(self, file_path, step_name):
        self.file_path = file_path
        self.step_name = step_name

        self.current_train_loss = None

        self.epoch_list = []
        self.train_loss_list = []
        self.val_loss_list = []
        self.val_f1_macro_list = []
        self.val_f1_micro_list = []
        self.val_pearson_mean_list = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None and "loss" in logs and "eval_loss" not in logs:
            self.current_train_loss = float(logs["loss"])

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is None:
            return

        epoch = int(round(float(metrics.get("epoch", state.epoch))))

        train_loss = (
            self.current_train_loss
            if self.current_train_loss is not None
            else ""
        )

        val_loss = float(metrics.get("eval_loss", 0.0))
        val_f1_macro = float(metrics.get("eval_f1_macro", 0.0))
        val_f1_micro = float(metrics.get("eval_f1_micro", 0.0))
        val_pearson_mean = float(metrics.get("eval_pearson_mean", 0.0))

        self.epoch_list.append(epoch)
        self.train_loss_list.append(train_loss)
        self.val_loss_list.append(val_loss)
        self.val_f1_macro_list.append(val_f1_macro)
        self.val_f1_micro_list.append(val_f1_micro)
        self.val_pearson_mean_list.append(val_pearson_mean)

        with open(self.file_path, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)

            writer.writerow([])
            writer.writerow([f"{self.step_name} EPOCH {epoch}"])

            writer.writerow([
                "epoch",
                "train_loss",
                "val_loss",
                "val_f1_macro",
                "val_f1_micro",
                "val_pearson_mean"
            ])

            for i in range(len(self.epoch_list)):
                writer.writerow([
                    self.epoch_list[i],
                    self.train_loss_list[i],
                    self.val_loss_list[i],
                    self.val_f1_macro_list[i],
                    self.val_f1_micro_list[i],
                    self.val_pearson_mean_list[i]
                ])

        print(f"{self.step_name} epoch {epoch} results saved.")


# =========================================================
# 16. FINAL HIERARCHICAL TEST FUNCTION
# =========================================================
def evaluate_hierarchical_on_test(step1_trainer, step2_trainer):
    pred_step1 = step1_trainer.predict(test_step1_ds)

    probs_step1 = 1 / (1 + np.exp(-pred_step1.predictions))
    pred_emotions = (probs_step1 >= 0.5).astype(int)

    pred_step2 = step2_trainer.predict(test_step2_ds)

    logits_step2 = pred_step2.predictions
    true_intensities = pred_step2.label_ids

    probs_step2 = torch.softmax(
        torch.tensor(logits_step2),
        dim=-1
    ).numpy()

    pred_intensities = np.argmax(probs_step2, axis=-1)

    final_pred_intensities = pred_intensities * pred_emotions

    true_binary_15 = np.zeros(
        (true_intensities.shape[0], len(LABELS)),
        dtype=int
    )

    pred_binary_15 = np.zeros(
        (true_intensities.shape[0], len(LABELS)),
        dtype=int
    )

    prob_binary_15 = np.zeros(
        (true_intensities.shape[0], len(LABELS)),
        dtype=float
    )

    for emotion_idx, emotion in enumerate(EMOTIONS):
        for level in [1, 2, 3]:
            col_idx = emotion_idx * 3 + (level - 1)

            true_binary_15[:, col_idx] = (
                true_intensities[:, emotion_idx] == level
            ).astype(int)

            pred_binary_15[:, col_idx] = (
                final_pred_intensities[:, emotion_idx] == level
            ).astype(int)

            prob_binary_15[:, col_idx] = (
                probs_step1[:, emotion_idx]
                * probs_step2[:, emotion_idx, level]
            )

    test_f1_macro = f1_score(
        true_binary_15,
        pred_binary_15,
        average="macro",
        zero_division=0
    )

    test_f1_micro = f1_score(
        true_binary_15,
        pred_binary_15,
        average="micro",
        zero_division=0
    )

    pearsons = []

    for i in range(true_binary_15.shape[1]):
        if np.std(true_binary_15[:, i]) == 0 or np.std(prob_binary_15[:, i]) == 0:
            pearsons.append(0.0)
        else:
            p, _ = pearsonr(true_binary_15[:, i], prob_binary_15[:, i])
            pearsons.append(0.0 if np.isnan(p) else float(p))

    test_pearson_mean = float(np.mean(pearsons))

    report_dict = classification_report(
        true_binary_15,
        pred_binary_15,
        target_names=LABELS,
        zero_division=0,
        output_dict=True
    )

    return {
        "step2_test_loss": pred_step2.metrics.get("test_loss", ""),
        "test_f1_macro": test_f1_macro,
        "test_f1_micro": test_f1_micro,
        "test_pearson_mean": test_pearson_mean,
        "classwise_report": report_dict
    }


# =========================================================
# 17. TRAIN ONE HIERARCHICAL FOLD
# =========================================================
def train_one_fold(fold_number):
    print("\n================================")
    print(f"TRAINING HIERARCHICAL FOLD {fold_number}")
    print("================================")

    set_seed(SEED + fold_number)

    fold_train_df = fold_data[fold_number]["train_df"]
    fold_val_df = fold_data[fold_number]["val_df"]

    print_fold_class_counts(fold_number)
    plot_selected_fold(fold_number)

    LOG_FILE = (
        f"{OUTPUT_DIR}/"
        f"RoBERTa_Hierarchical_Fold_{fold_number}.csv"
    )

    with open(LOG_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        writer.writerow(["model", MODEL_NAME])
        writer.writerow(["fold", fold_number])
        writer.writerow(["method", "two-step hierarchical classification"])
        writer.writerow(["learning_rate", 2e-5])
        writer.writerow(["step1_batch_size", 8])
        writer.writerow(["step2_batch_size", 8])
        writer.writerow(["step1_max_epochs", 10])
        writer.writerow(["step2_max_epochs", 10])
        writer.writerow(["train_samples", len(fold_train_df)])
        writer.writerow(["validation_samples", len(fold_val_df)])
        writer.writerow([])

        writer.writerow(["CLASS DISTRIBUTION"])
        writer.writerow(["class", "train_count", "validation_count"])

        train_counts = get_15_label_counts(fold_train_df)
        val_counts = get_15_label_counts(fold_val_df)

        for label in LABELS:
            writer.writerow([
                label,
                train_counts[label],
                val_counts[label]
            ])

    train_step1_ds = prepare_step1_dataset(fold_train_df)
    val_step1_ds = prepare_step1_dataset(fold_val_df)

    train_step2_ds = prepare_step2_dataset(fold_train_df)
    val_step2_ds = prepare_step2_dataset(fold_val_df)

    # =====================================================
    # STEP 1 TRAINING - EMOTION DETECTION
    # =====================================================
    print("\nStarting Step 1: Emotion Detection")

    model_step1 = RobertaForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(EMOTIONS),
        problem_type="multi_label_classification"
    )

    args_step1 = TrainingArguments(
        output_dir=f"/content/roberta_hier_fold_{fold_number}_step1",
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=10,
        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=1,
        report_to="none",
        fp16=torch.cuda.is_available(),
        seed=SEED + fold_number
    )

    step1_logger = SaveEpochLogger(
        file_path=LOG_FILE,
        step_name="STEP 1"
    )

    trainer_step1 = Trainer(
        model=model_step1,
        args=args_step1,
        train_dataset=train_step1_ds,
        eval_dataset=val_step1_ds,
        data_collator=data_collator,
        processing_class=tokenizer,
        compute_metrics=compute_metrics_step1,
        callbacks=[
            step1_logger,
            StopOnValLossAfterMinEpoch(min_epoch=3, patience=1)
        ]
    )

    start1 = time.time()
    trainer_step1.train()
    end1 = time.time()

    with open(LOG_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        writer.writerow([])
        writer.writerow(["STEP 1 TRAINING SUMMARY"])
        writer.writerow(["step1_task", "emotion detection"])
        writer.writerow(["step1_training_time_seconds", end1 - start1])
        writer.writerow(["step1_best_checkpoint", trainer_step1.state.best_model_checkpoint])
        writer.writerow(["step1_best_metric", trainer_step1.state.best_metric])

    # =====================================================
    # STEP 2 TRAINING - INTENSITY DETECTION
    # =====================================================
    print("\nStarting Step 2: Intensity Detection")

    model_step2 = RobertaStep2IntensityModel()

    args_step2 = TrainingArguments(
        output_dir=f"/content/roberta_hier_fold_{fold_number}_step2",
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=10,
        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=1,
        report_to="none",
        fp16=torch.cuda.is_available(),
        seed=SEED + fold_number
    )

    step2_logger = SaveEpochLogger(
        file_path=LOG_FILE,
        step_name="STEP 2"
    )

    trainer_step2 = Trainer(
        model=model_step2,
        args=args_step2,
        train_dataset=train_step2_ds,
        eval_dataset=val_step2_ds,
        data_collator=data_collator,
        compute_metrics=compute_metrics_step2,
        callbacks=[
            step2_logger,
            StopOnValLossAfterMinEpoch(min_epoch=3, patience=1)
        ]
    )

    start2 = time.time()
    trainer_step2.train()
    end2 = time.time()

    with open(LOG_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        writer.writerow([])
        writer.writerow(["STEP 2 TRAINING SUMMARY"])
        writer.writerow(["step2_task", "intensity detection"])
        writer.writerow(["step2_training_time_seconds", end2 - start2])
        writer.writerow(["step2_best_checkpoint", trainer_step2.state.best_model_checkpoint])
        writer.writerow(["step2_best_metric", trainer_step2.state.best_metric])

    # =====================================================
    # FINAL TEST AFTER THIS FOLD
    # =====================================================
    test_results = evaluate_hierarchical_on_test(
        step1_trainer=trainer_step1,
        step2_trainer=trainer_step2
    )

    report_dict = test_results["classwise_report"]

    with open(LOG_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        writer.writerow([])
        writer.writerow(["FINAL HIERARCHICAL TEST RESULTS FOR THIS FOLD"])
        writer.writerow(["metric", "value"])
        writer.writerow(["step2_test_loss", test_results["step2_test_loss"]])
        writer.writerow(["test_f1_macro", test_results["test_f1_macro"]])
        writer.writerow(["test_f1_micro", test_results["test_f1_micro"]])
        writer.writerow(["test_pearson_mean", test_results["test_pearson_mean"]])

        writer.writerow([])
        writer.writerow(["CLASSWISE HIERARCHICAL TEST RESULTS"])
        writer.writerow([
            "class",
            "precision",
            "recall",
            "f1_score",
            "support"
        ])

        for label in LABELS:
            row = report_dict.get(label, {})

            writer.writerow([
                label,
                row.get("precision", ""),
                row.get("recall", ""),
                row.get("f1-score", ""),
                row.get("support", "")
            ])

    print("\nFold completed.")
    print("Test macro F1:", test_results["test_f1_macro"])
    print("Test micro F1:", test_results["test_f1_micro"])
    print("Test Pearson:", test_results["test_pearson_mean"])
    print("Saved at:", LOG_FILE)

    return LOG_FILE




In [ ]:
# =========================================================
# 18. MANUAL USAGE
# =========================================================
# Run one fold at a time:
#
fold_1_file = train_one_fold(1)


In [ ]:
fold_2_file = train_one_fold(2)


In [ ]:
fold_3_file = train_one_fold(3)


In [ ]:
fold_4_file = train_one_fold(4)


In [ ]:
fold_5_file = train_one_fold(5)